# DYSANOS
___

___
### ML-SANOS

**Summary**

SANOS can be viewed as the mapping
$$
(q, W, \hat{K}, \hat{T}, \mu) \xrightarrow{SANOS} C
$$
from which we want to learn the martingale density $q$ and the variance $W$.

Since $q$ is subject to linear constraints, it is hard to use ML on it. The idea of ML-SANOS is to create the mapping
$$
\small 
(h;z,\hat{T},\mu) \xrightarrow{\tiny NQ} (x;z,\hat{T};\mu) \xrightarrow{\tiny ML-SANOS} 
\\\\ 
(\Sigma, W; \hat{K},\hat{T}; \mu) \xrightarrow{\tiny DLV} (q, W; \hat{K}, \hat{T}; \mu) \xrightarrow{\tiny SANOS} C
$$
Then, in the new parametrization, $x$ lives in the unrestricted space and is therefore well suited for ML/AI based learning.

To simplify, the chain looks like:
$$
\small 
(h^{20}) \xrightarrow{\tiny NQ} (x^{160}) \xrightarrow{\tiny ML-SANOS} 
(\Sigma, W) \xrightarrow{\tiny DLV} (q) \xrightarrow{\tiny SANOS} C_{\tiny SANOS}, \, \text{\tiny optimized min loss: Bid-Ask}
$$

In other words, there is a direct mapping from $(x)$ to $(\Sigma, W)$ to $(q)$ to $C_{\tiny SANOS}$. The goal is instead of calibrating $q$ directly using LP solver, we train $x$ using batch gradient descent, optimized using the direct mapping from $x$ to $C_{\tiny SANOS}$ (against loss function). This way, the process is differentiable end-to-end, and is faster through GPU batching. Then, since all is differentiable, we can add $h$ in front of $x$ and train the whole thing. $h$ is only added for dimension reduction, but $x$ alone is sufficient for the whole process.

After training, we are left with 2 things:
- A time serie {${h_t}$}$_{t=1}^{N_{\tiny training}}$, $h_t \in \mathbb{R}^{20}$ for each training day $t$
- A single shared decoder $\theta$ from $NQ_{\theta}$ (that leads to a valid SANOS surface), valid across all days

For simulation, a new $h_t$ generates a new valid surface.

The authors analyse the time serie {$h_t$} with a standard model such as PCA AR(1). Other models are subject to further research.

**DLV** (Discrete local volatility)

**ML-SANOS**

**NQ** 

___
### DYSANOS

**Remark**

Surface are normalized, so we can split market generation between option surfaces and spot. (We generally want to sample option surfaces less frequently than spot)

In the paper, they use a stationary distribution for option surface parameters.

**Theorem 3.1.** Any generative non-degenerate model for sequences $x_t=(s_t,h_t)$ gives rise to a dynamic generative ML-SANOS model which has, for each time step $t$, a spot price $S_t$ and a call price function $C_t$, which is free of static arbitrage and smooth.

**Section 3.4** Static absence of arbitrage of each decoded surface does not by itself imply absence of dynamic arbitrage between trading times. We here brute-force test a subset of possible arbitrage-strategies and will show that while DYSANOS does not have arbitrage opportunities when traded options expire at their expiry – as expected –, it does have arbitrage opportunities if the same options can be sold at a later point for their mark to market.

**DYSANOS Model Choice**

The rest of the **Section 3** is about modeling choice of the authors and properties of there chosen model.

**Baseline Model**

Assume we are given historic samples of ML-SANOS surface states $\tilde{h}_t = (\tilde{h}^1_t, \ldots, \tilde{h}^{n_h}_t) \in \mathbb{R}^{n_h}$ for each historic date $t \in {t_1, \ldots, t_{n_t}}$. We also observe log-spot $\tilde{s}_t := \log S_t$. We use the tilde to distinguish real observed data from simulated data.

*Continuous-time PCA-AR(1)*
$$
dh_t = \kappa (m - h_t) \ dt + \Sigma_h \ dW_t^h
$$
for $m \in \mathbb{R}^{n_h}$, $\kappa \in \mathbb{R}^{n_h, n_h}$, $\Sigma_h \in \mathbb{R}^{n_h,n_\alpha}$ and a Brownian motion $W_t^h \in \mathbb{R}^{n_\alpha}$. $n_h$ is the dimention of thre vector $h_t$, $n_{\alpha}$ is the number of PCA factors.

Notes: need all eigenvalues of $\kappa$ to be positive $\rightarrow$ stationary and invertible.

*Log-spot* (fitted afterward)
$$
ds_t = \mu dt + \beta^{\prime} dW_t^h + \varsigma dW_t^s
$$
where $\mu \in \mathbb{R}$ is an intercept, $\beta \in \mathbb{R}^{n_h}$, $\varsigma \in \mathbb{R}$ and $dW_t^s \in \mathbb{R}$ is an independant Brownian motion. This preserves the autonomous surface dynamics and captures contemporaneous spot–surface dependence through $\beta$.

**Simulation**

1. Draw randomly $h_0$ from ${\tilde h}_{t\in{1,\dots,n_t}}$ or the invariant distribution.

2. Generate the surface path using the Euler discretization
    $$
    h_{t+dt} = h_t + \kappa(m - h_t)\Delta_t + \Sigma_h Z_t\sqrt{\Delta_t}
    $$
    where $\Delta_t$ is the normalized time increment. The noise term is defined as mixture
    $$
    Z_t = \sqrt{1-b^2} \ \tilde\alpha_{I_t(\omega)} + b \ Y_t,
    $$
    between a resampled standardized historic PCA score and an independent standard normal $Y_t \sim \mathcal{N}(0, I_{n_\alpha})$ for bandwidth $b\in[0,1]$. Here $I_t(\omega)$ is sampled from the indices of the historic transitions. Hence $b=0$ gives empirical innovation resampling, while $b=1$ gives a Gaussian innovation with the same fitted covariance.

3. Conditional on the complete surface path, generate log-spot using 
    $$
    ds_t = \mu dt + \beta^{\prime} dW_t^h + \varsigma dW_t^s
    $$

___
### Implement DYSANOS

<div style="background-color: #8bd3aa; padding: 10px; border-left: 4px solid #07ff24; color: black;">
⚠️ <b>TO DO</b> 
</div>

- Need time serie of full surface

___
### TO DO

**Paper to read**
- Discrete local volatility
- Dupire